In [ ]:
!pip install transformers torch scikit-learn -q

import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils import resample
from google.colab import drive

def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(s):
        if s < num_warmup_steps:
            return s / max(1, num_warmup_steps)
        return max(0.0, (num_training_steps-s) / max(1, num_training_steps-num_warmup_steps))
    return LambdaLR(optimizer, lr_lambda)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("All imported!")

Device: cuda
All imported!


In [ ]:
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Multimodal/multibully_with_prompts.csv")

SEED = 42

# ── TASK 2 CHANGE — split on sentiment_label instead of bully_label ──
train_val, test_df = train_test_split(df, test_size=0.20,
                     stratify=df["sentiment_label"], random_state=SEED)
train_df, val_df   = train_test_split(train_val, test_size=0.125,
                     stratify=train_val["sentiment_label"], random_state=SEED)

# Oversample minority classes in training
# Positive (607) is the rarest — oversample to match Negative (most common)
from sklearn.utils import resample

neg  = train_df[train_df["sentiment_label"]==0]  # Negative
neu  = train_df[train_df["sentiment_label"]==1]  # Neutral
pos  = train_df[train_df["sentiment_label"]==2]  # Positive
n    = max(len(neg), len(neu))

train_balanced = pd.concat([
    neg,
    resample(neu, replace=True, n_samples=n, random_state=42),
    resample(pos, replace=True, n_samples=n, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train (balanced): {len(train_balanced)}")
print(f"Val             : {len(val_df)}")
print(f"Test            : {len(test_df)}")
print(f"\nSentiment distribution in test:")
print(test_df["sentiment_label"].value_counts().rename({0:"Negative",1:"Neutral",2:"Positive"}))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train (balanced): 5577
Val             : 580
Test            : 1159

Sentiment distribution in test:
sentiment_label
Negative    532
Neutral     506
Positive    121
Name: count, dtype: int64


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model     = AutoModelForMaskedLM.from_pretrained("xlm-roberta-base").to(device)

# ── TASK 2 CHANGE — sentiment verbalizer words ──
negative_id = tokenizer.convert_tokens_to_ids(tokenizer.tokenize("negative")[0])
neutral_id  = tokenizer.convert_tokens_to_ids(tokenizer.tokenize("neutral")[0])
positive_id = tokenizer.convert_tokens_to_ids(tokenizer.tokenize("positive")[0])

print(f"Model ready on  : {device}")
print(f"negative token id: {negative_id}")
print(f"neutral  token id: {neutral_id}")
print(f"positive token id: {positive_id}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready on  : cuda
negative token id: 40907
neutral  token id: 62276
positive token id: 24491


In [ ]:
# ── TASK 2 CHANGE — use sentiment_label and prompt_C ──
# Template C "The image shows [caption]. The sentiment of this image is <mask>."
# is the best fit for sentiment task

PROMPT_COL  = "prompt_A"
LABEL_COL   = "sentiment_label"

class PromptDataset(Dataset):
    def __init__(self, df, col=PROMPT_COL, label=LABEL_COL):
        self.prompts = df[col].tolist()
        self.labels  = df[label].tolist()
    def __len__(self): return len(self.prompts)
    def __getitem__(self, idx):
        enc = tokenizer(self.prompts[idx], return_tensors="pt",
                        truncation=True, max_length=128, padding="max_length")
        return {"input_ids"     : enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label"         : self.labels[idx]}

train_loader = DataLoader(PromptDataset(train_balanced), batch_size=8, shuffle=True)
val_loader   = DataLoader(PromptDataset(val_df),         batch_size=8, shuffle=False)
test_loader  = DataLoader(PromptDataset(test_df),        batch_size=8, shuffle=False)

print(f"Using template : {PROMPT_COL}")
print(f"Using label    : {LABEL_COL}")
print(f"Train batches  : {len(train_loader)}")

Using template : prompt_A
Using label    : sentiment_label
Train batches  : 698


In [ ]:
def evaluate(loader):
    model.eval()
    preds, trues = [], []

    for batch in loader:
        ids  = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)

        with torch.no_grad():
            logits = model(input_ids=ids, attention_mask=attn).logits

        for i in range(ids.shape[0]):
            pos = (ids[i]==tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
            if len(pos)==0:
                preds.append(1)  # default to neutral
            else:
                ml = logits[i, pos[0], :]
                # Pick highest score among 3 sentiment words
                scores = {
                    0: ml[negative_id].item(),
                    1: ml[neutral_id].item(),
                    2: ml[positive_id].item()
                }
                preds.append(max(scores, key=scores.get))

        trues.extend(batch["label"].tolist())

    return f1_score(trues, preds, average="macro"), trues, preds

print("Evaluate function ready!")

Evaluate function ready!


In [ ]:
NUM_EPOCHS = 7
optimizer  = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scheduler  = get_linear_schedule_with_warmup(
    optimizer, 100, (len(train_loader)//2)*NUM_EPOCHS
)

best_f1, best_state = 0, None
scaler = torch.amp.GradScaler("cuda") if device=="cuda" else None

print(f"Training Task 2 — Sentiment | {PROMPT_COL} | {NUM_EPOCHS} epochs")
print("-" * 55)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, batch in enumerate(train_loader):
        ids    = batch["input_ids"].to(device)
        attn   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # Map sentiment labels to token ids
        target = torch.zeros_like(labels)
        target[labels==0] = negative_id
        target[labels==1] = neutral_id
        target[labels==2] = positive_id

        lbl = torch.full(ids.shape, -100, device=device)
        lbl[(ids==tokenizer.mask_token_id)] = target.repeat_interleave(
            (ids==tokenizer.mask_token_id).sum(dim=1))

        if scaler:
            with torch.amp.autocast("cuda"):
                loss = model(input_ids=ids, attention_mask=attn,
                             labels=lbl).loss / 2
            scaler.scale(loss).backward()
        else:
            loss = model(input_ids=ids, attention_mask=attn,
                         labels=lbl).loss / 2
            loss.backward()

        total_loss += loss.item() * 2

        if (i+1) % 2 == 0:
            if scaler:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        del ids, attn, labels
        if device=="cuda": torch.cuda.empty_cache()

    val_f1, _, _ = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Val F1: {val_f1*100:.2f}%")

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  ★ Best saved!")

model.load_state_dict(best_state)
print(f"\nDone! Best Val F1: {best_f1*100:.2f}%")

Training Task 2 — Sentiment | prompt_A | 7 epochs
-------------------------------------------------------
Epoch 1/7 | Loss: 1.5257 | Val F1: 49.48%
  ★ Best saved!
Epoch 2/7 | Loss: 0.7630 | Val F1: 48.45%
Epoch 3/7 | Loss: 0.5420 | Val F1: 50.73%
  ★ Best saved!
Epoch 4/7 | Loss: 0.3939 | Val F1: 46.32%
Epoch 5/7 | Loss: 0.2917 | Val F1: 50.43%
Epoch 6/7 | Loss: 0.2179 | Val F1: 50.79%
  ★ Best saved!
Epoch 7/7 | Loss: 0.1783 | Val F1: 51.43%
  ★ Best saved!

Done! Best Val F1: 51.43%


In [ ]:
test_f1, y_true, y_pred = evaluate(test_loader)

print(f"=== TASK 2 RESULTS — SENTIMENT ===")
print(f"Macro F1 : {test_f1*100:.2f}%")
print(classification_report(y_true, y_pred,
      target_names=["Negative", "Neutral", "Positive"]))

=== TASK 2 RESULTS — SENTIMENT ===
Macro F1 : 49.60%
              precision    recall  f1-score   support

    Negative       0.57      0.71      0.63       532
     Neutral       0.57      0.39      0.46       506
    Positive       0.35      0.45      0.40       121

    accuracy                           0.54      1159
   macro avg       0.50      0.52      0.50      1159
weighted avg       0.55      0.54      0.53      1159



In [ ]:
import os

save_path = "/content/drive/MyDrive/Multimodal/xlmr_task2_sentiment/"
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

pred_path = "/content/drive/MyDrive/Multimodal/predictions_task2.csv"
test_df_save = test_df.copy()
test_df_save["pred_sentiment"] = y_pred
test_df_save["true_sentiment"] = y_true
test_df_save.to_csv(pred_path, index=False)

print(f"Model saved      : {save_path}")
print(f"Predictions saved: {pred_path}")
print(f"Final Macro F1   : {test_f1*100:.2f}%")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved      : /content/drive/MyDrive/Multimodal/xlmr_task2_sentiment/
Predictions saved: /content/drive/MyDrive/Multimodal/predictions_task2.csv
Final Macro F1   : 49.60%
